# Phase 4 Lab — Reference Solution

**Phase:** Statistics and Mathematics  
**Scenario:** A team compares a redesigned process with the existing process using completion-time samples.

**Deliverable:** A statistically defensible analysis including design, uncertainty, test, effect size, assumptions, and practical recommendation.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Define population, estimand, null, alternative, and minimum meaningful effect.
2. Visualize distributions and inspect independence/design assumptions.
3. Calculate descriptive summaries and confidence intervals.
4. Run an appropriate comparison test.
5. Calculate standardized and raw effect sizes.
6. Perform a sensitivity or bootstrap analysis.
7. Separate statistical and practical conclusions.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
from scipy import stats
rng_local=np.random.default_rng(42)
control=rng_local.lognormal(mean=np.log(42),sigma=.28,size=120)
redesign=rng_local.lognormal(mean=np.log(38),sigma=.28,size=118)

def bootstrap_mean_difference(a,b,repetitions=5000):
    differences=[]
    for _ in range(repetitions):
        differences.append(rng_local.choice(a,len(a),replace=True).mean()-
                           rng_local.choice(b,len(b),replace=True).mean())
    return np.quantile(differences,[.025,.5,.975])

difference=redesign.mean()-control.mean()
t,p=stats.ttest_ind(redesign,control,equal_var=False)
pooled=np.sqrt(((len(redesign)-1)*redesign.var(ddof=1)+(len(control)-1)*control.var(ddof=1))/
               (len(redesign)+len(control)-2))
d=difference/pooled
ci=bootstrap_mean_difference(redesign,control)
summary=pd.DataFrame({
    "group":["control","redesign"],
    "n":[len(control),len(redesign)],
    "mean":[control.mean(),redesign.mean()],
    "median":[np.median(control),np.median(redesign)],
    "std":[control.std(ddof=1),redesign.std(ddof=1)],
})
display(summary.round(3))
print(f"Raw mean difference (redesign-control): {difference:.3f}")
print(f"Welch t={t:.3f}, p={p:.4f}, Cohen's d={d:.3f}")
print("Bootstrap 95% interval for mean difference:",ci.round(3))
print("Decision must compare this interval with the predeclared operationally meaningful reduction.")

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.